# 05 · CXR embeddings — frozen RAD-DINO CLS, cached per image

Extracts the **768-d CLS token** from the LP-FT backbone (`raddino_mld_db_ft_best.pt`)
for every image, caches it keyed by `dicom_id`, then pools to one `x_cxr` per study.
This is the frozen image feature `06` conditions on with FiLM.

**Why this notebook is safe to run frozen** (04b · C9-bis): the train−val AUPRC gap
came in at **+0.042**, well under the 0.15 line at which train embeddings would be
optimistic enough to distort the fusion ablation. The residual optimism sits on
Pneumothorax / Lung_Lesion, and it only bites if fusion *selects* on train — it does
not (thresholds and FiLM identity-init are tuned on val). So: extract once, freeze,
no cross-fitting.

**Two design points, both taken from §B2 of the plan:**

| | plan §B2 / §B3 | here |
|---|---|---|
| cache granularity | `.npy [N,768]` + `dicom_id` index; join at train time | **per image**, keyed by `dicom_id` |
| study pooling | *"flag; default mean-pool over frontal"* | a **runtime flag**, default `w_f=0.7 / w_l=0.3` (Tuan) |

Caching per image (not pre-pooled) is what lets `06` run the **view/pooling ablation**
the plan asks for — mean-frontal, 7:3, attention-pool — without re-extracting the
whole cohort (~40 min). The 7:3 is just this notebook's default; the embeddings
underneath are pooling-agnostic.

**Backbone only.** We load just the `backbone.*` weights into a bare `AutoModel`
and discard the ML-Decoder head — CLS does not pass through it — so 05 needs neither
`ml_decoder.py` nor the torch-2.x shims 04b carried.

**Prereq:** 04b finished (`raddino_mld_db_ft_best.pt` on Drive) and 04a's tar shards
present (`cxr_560_shards`). This notebook stages the images itself in **C1** — no
dependency on having run 04b in the same VM. `/content` is wiped on every restart, so
C1 always runs (instant when the local copy is already warm).


## C0 · Config

In [1]:
import os, io, time, json, math, glob, shutil, tarfile
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR  = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
CKPT_DIR  = os.path.join(DATA_DIR, 'lpft_v2')
CACHE_DIR = os.path.join(DATA_DIR, 'cxr_cls_cache'); os.makedirs(CACHE_DIR, exist_ok=True)
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards') # tar shards from 04a (on Drive)
LOCAL_IMG = '/content/cxr_local'                      # extracted here; WIPED on VM restart -> C1 re-stages
BEST_CKPT = os.path.join(CKPT_DIR, 'raddino_mld_db_ft_best.pt')

# ---- pooling flag (per image cache stays pooling-agnostic; this only sets the default
#      x_cxr written to disk. 06 can re-pool from the per-image cache for free.) ----
W_F, W_L  = 0.7, 0.3
assert abs(W_F + W_L - 1.0) < 1e-9

EXTRACT_BATCH = 96                                    # eval-only, no grad -> big batch
NUM_WORKERS   = 8
BLOCK         = 10000                                 # images per cache block (resume unit)
DTYPE_NP      = np.float32                            # plan §B2: float32 [N,768]
SEED          = 42
torch.manual_seed(SEED); np.random.seed(SEED)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.cuda.get_device_name(0) if dev=='cuda' else 'NO GPU')
assert os.path.exists(BEST_CKPT), f'{BEST_CKPT} not found -> run 04b first'
print('checkpoint:', BEST_CKPT)

Mounted at /content/drive
NVIDIA RTX PRO 6000 Blackwell Server Edition
checkpoint: /content/drive/MyDrive/MIMIC_CXR/planB_out/lpft_v2/raddino_mld_db_ft_best.pt


## C1 · Stage the shards to local disk

The images live on Drive as **tar shards** (`cxr_560_shards`, ~17 GB); `/content` is
the VM's scratch disk and is **wiped on every restart**. So this must run in *this*
notebook's session even if 04b already staged them in a previous one — a fresh
runtime starts empty. Identical to 04b · C1 (copy → extract → drop the tar, one shard
at a time; a `.done_` marker written *after* extract makes it resumable and skips
what is already local).

Copying is CPU/network only — **no GPU needed for this cell** (~8 min for a full
set). If the local images survive from an earlier cell in the same session, this
finishes instantly.

In [2]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
assert tars, f'no shards in {SHARD_DIR} -> run 04a first'
print(f'shards on Drive: {len(tars)}   already local: {len(os.listdir(LOCAL_IMG)):,} files')
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards to stage: {len(todo)} / {len(tars)}   (~8 min for a full set; instant if warm)')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()   # marker AFTER extract
n_local = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'local images: {n_local:,}')

shards on Drive: 347   already local: 0 files
shards to stage: 347 / 347   (~8 min for a full set; instant if warm)


staging shards:   0%|          | 0/347 [00:00<?, ?shard/s]

/tmp/ipykernel_1823/2768569168.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)


local images: 346,578


## C2 · Cohort + view map — same `items` table as 04b

`items` is one row per (study, image) for frontal/lateral views only, carrying
`view_group` and `split`. Identical construction to 04b · C4 so the `dicom_id`
universe matches exactly — the cache we build here is the one 06 will index.

Missing local files are dropped **loudly**: a silent shrink here would later look
like studies with no embedding. Same <1% guard as 04b.

In [3]:
_meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS  = _meta['labels']; assert len(LABELS) == 13
base    = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'))
vm      = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
items   = (vm[vm.view_group.isin(['frontal','lateral'])]
             .merge(base[['study_id','split']], on='study_id', how='inner'))

before  = len(items)
have    = items.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
lost    = items[~have]
items   = items[have].sort_values('dicom_id').reset_index(drop=True)   # stable order = stable blocks
print(f'images: {len(items):,} / {before:,}   dropped {len(lost):,} with no local file')
assert len(lost) / before < 0.01, \
    (f'{len(lost):,} images ({len(lost)/before:.2%}) missing from {LOCAL_IMG} -> '
     'run C1 first to stage the shards (100% missing = fresh VM, C1 not run yet)')

print('by split :', dict(items.drop_duplicates("study_id").split.value_counts()))
print('by view  :', dict(items.view_group.value_counts()))
print('studies  :', items.study_id.nunique(), '| images', len(items))

images: 346,578 / 346,578   dropped 0 with no local file
by split : {'train': np.int64(147791), 'val': np.int64(31670), 'test': np.int64(31669)}
by view  : {'frontal': np.int64(234727), 'lateral': np.int64(111851)}
studies  : 211130 | images 346578


## C3 · Backbone — `backbone.*` weights only

The checkpoint holds `backbone.*` and `head.*`. CLS is the backbone's output before
the head, so we load only the backbone weights into a plain `AutoModel` (`strict=True`
against the filtered keys — a mismatch means the checkpoint is not what we think it
is, and should fail here, not silently half-load).

In [4]:
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
from PIL import Image

backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()

sd    = torch.load(BEST_CKPT, map_location=dev, weights_only=False)['model']
bb_sd = {k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}
assert bb_sd, 'no backbone.* keys in checkpoint'
backbone.load_state_dict(bb_sd, strict=True)          # exact match or bust
D = backbone.config.hidden_size; assert D == 768
print(f'loaded {len(bb_sd)} backbone tensors | hidden {D}')

proc       = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']
MEAN, STD  = proc.image_mean[0], proc.image_std[0]
# eval_tf ONLY -- extraction must be deterministic; no RandAug (that is for training)
eval_tf = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
print(f'MODEL_SIZE {MODEL_SIZE} | mean {MEAN} std {STD}')

class CXRImg(Dataset):
    """image + its row index; no labels needed for extraction."""
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        d  = self.df.dicom_id.iloc[i]
        im = Image.open(f'{LOCAL_IMG}/{d}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))         # RAD-DINO wants 3 identical channels
        return eval_tf(im), i

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

loaded 223 backbone tensors | hidden 768


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE 518 | mean 0.5307 std 0.2583


## C4 · Extract CLS → per-image cache, **resumable by block**

CLS for all ~347k images is one forward pass each: ~40 min. Colab will disconnect,
so the cache is written in **blocks of 10 000 images** (04a's shard-resume idea): a
finished block is a `.npy` on Drive; a re-run skips every block already present and
only computes the gaps. Blocks are keyed by position in the `dicom_id`-sorted `items`,
so the same image always lands in the same block.

Each block writes `cls_block_XXXXX.npy [n,768]` + `..._ids.parquet` (its `dicom_id`s
in order) via `.tmp` → `os.replace`, so a kill mid-write never leaves a half block.

In [5]:
@torch.no_grad()                                      # plain fn (not a generator): no_grad is unambiguous
def extract_block(sub, pbar):
    dl  = DataLoader(CXRImg(sub), batch_size=EXTRACT_BATCH, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True)
    out = np.zeros((len(sub), D), DTYPE_NP)
    for xb, idx in dl:
        with torch.autocast('cuda', dtype=torch.bfloat16):
            cls = backbone(pixel_values=xb.to(dev, non_blocking=True)).last_hidden_state[:, 0]
        out[idx.numpy()] = cls.float().cpu().numpy()
        pbar.update(len(idx))
    return out

def blk_paths(b): return (os.path.join(CACHE_DIR, f'cls_block_{b:05d}.npy'),
                          os.path.join(CACHE_DIR, f'cls_block_{b:05d}_ids.parquet'))

N      = len(items)
n_blk  = math.ceil(N / BLOCK)
todo   = [b for b in range(n_blk) if not (os.path.exists(blk_paths(b)[0])
                                          and os.path.exists(blk_paths(b)[1]))]
print(f'{N:,} images -> {n_blk} blocks of {BLOCK}   ({n_blk-len(todo)} already cached, {len(todo)} to do)')

t0 = time.time()
pbar = tqdm(total=sum(min(BLOCK, N-b*BLOCK) for b in todo), unit='img', desc='extract CLS')
for b in todo:
    sub = items.iloc[b*BLOCK:(b+1)*BLOCK]
    out = extract_block(sub, pbar)
    npy, ids = blk_paths(b)
    np.save(npy + '.tmp.npy', out); os.replace(npy + '.tmp.npy', npy)       # atomic
    sub[['dicom_id']].reset_index(drop=True).to_parquet(ids + '.tmp'); os.replace(ids + '.tmp', ids)
    pbar.set_postfix_str(f'block {b+1}/{n_blk} | {pbar.n/max(time.time()-t0,1e-9):.0f} img/s')
pbar.close()
print(f'done. {n_blk} blocks in {CACHE_DIR}')

346,578 images -> 35 blocks of 10000   (0 already cached, 35 to do)


extract CLS:   0%|          | 0/346578 [00:00<?, ?img/s]

done. 35 blocks in /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_cls_cache


## C5 · Assemble the per-image cache

Concatenate the blocks into one `cxr_cls.npy [N,768]` + `cxr_cls_index.parquet`
(`dicom_id → row`). This is the artifact 06 indexes by `dicom_id`. Kept as `.npy`
(plan §B2: *".npy over 768 parquet columns at this width"*).

In [6]:
blocks = sorted(glob.glob(os.path.join(CACHE_DIR, 'cls_block_*.npy')))
assert len(blocks) == math.ceil(N / BLOCK), \
    f'{len(blocks)} block .npy files but expected {math.ceil(N/BLOCK)} -> some block is missing, re-run C3'

CLS = np.concatenate([np.load(f) for f in blocks], axis=0).astype(DTYPE_NP)
ids = pd.concat([pd.read_parquet(f.replace('.npy', '_ids.parquet')) for f in blocks],
                ignore_index=True)
assert len(CLS) == len(ids) == N, (len(CLS), len(ids), N)
assert ids.dicom_id.tolist() == items.dicom_id.tolist(), \
    'block dicom order != items order -> a block was built against a different items table'
assert not np.isnan(CLS).any(), 'NaN in CLS cache'

np.save(os.path.join(CACHE_DIR, 'cxr_cls.npy'), CLS)
ids.assign(row=np.arange(N)).to_parquet(os.path.join(CACHE_DIR, 'cxr_cls_index.parquet'))
print(f'cxr_cls.npy  {CLS.shape}  {CLS.nbytes/1e9:.2f} GB')
print(f'CLS norm: mean {np.linalg.norm(CLS,axis=1).mean():.2f}  '
      f'std {np.linalg.norm(CLS,axis=1).std():.2f}')

cxr_cls.npy  (346578, 768)  1.06 GB
CLS norm: mean 14.89  std 1.46


## C6 · Pool to study level — `x_cxr`, default 7:3

$$x^{cxr}_{study}=\frac{w_f\,\overline{cls}_{frontal}+w_l\,\overline{cls}_{lateral}}
{w_f\,[\exists frontal]+w_l\,[\exists lateral]}$$

Same renormalising denominator as 04b's probability pooling, and for the same reason:
51.6% of studies are frontal-only, so a plain weighted **sum** would scale their
embedding by 0.7 and shift them off the manifold the both-view studies live on.
Mean **within** a view group first (so a study with 3 frontals does not out-vote one
with 1), then weight **across** groups.

Written as `.npy` + a `study_id/split` index — the `x_cxr` half of the §B3 data
contract. The per-image cache stays on disk, so 06 can re-pool at any `w_f` for free.

In [7]:
cls_df = pd.DataFrame(CLS, columns=[f'e{j}' for j in range(D)])
cls_df['study_id']   = items.study_id.values
cls_df['view_group'] = items.view_group.values

ecols = [f'e{j}' for j in range(D)]
studies = np.sort(items.study_id.unique())
fr = cls_df[cls_df.view_group=='frontal'].groupby('study_id')[ecols].mean()
la = cls_df[cls_df.view_group=='lateral'].groupby('study_id')[ecols].mean()
fr = fr.reindex(studies); la = la.reindex(studies)               # NaN where a group is absent
hf = fr.notna().all(1).to_numpy().astype(DTYPE_NP)               # has-frontal indicator
hl = la.notna().all(1).to_numpy().astype(DTYPE_NP)

Fv = np.nan_to_num(fr.to_numpy(DTYPE_NP)); Lv = np.nan_to_num(la.to_numpy(DTYPE_NP))
den = (W_F*hf + W_L*hl)[:, None]
assert (den > 0).all(), 'a study has neither frontal nor lateral -- impossible after C1 filter'
X   = (W_F*Fv*hf[:,None] + W_L*Lv*hl[:,None]) / den               # [n_study, 768]

idx = (base[base.study_id.isin(studies)][['study_id','split']]
         .set_index('study_id').reindex(studies).reset_index())
assert idx.study_id.tolist() == list(studies)
assert not np.isnan(X).any()

np.save(os.path.join(CACHE_DIR, 'study_x_cxr.npy'), X)
idx.assign(row=np.arange(len(studies)),
           w_f=W_F, w_l=W_L).to_parquet(os.path.join(CACHE_DIR, 'study_x_cxr_index.parquet'))
print(f'x_cxr  {X.shape}  (w_f={W_F}, w_l={W_L})')
print('composition:', dict(pd.Series(np.select([(hf>0)&(hl>0),(hf>0)&(hl==0),(hf==0)&(hl>0)],
                    ['both','frontal_only','lateral_only'],'none')).value_counts()))

x_cxr  (211130, 768)  (w_f=0.7, w_l=0.3)
composition: {'frontal_only': np.int64(109052), 'both': np.int64(101712), 'lateral_only': np.int64(366)}


## C7 · Sanity — coverage, split, no leaks in the index

Every cohort study must have exactly one `x_cxr`; the split labels must match `base`;
and standardization stats for 06 come from **train only** — checked here so a later
notebook cannot quietly fit them on val/test.

In [8]:
n_cohort = base.study_id.nunique()
print(f'studies in x_cxr : {len(studies):,} / {n_cohort:,} cohort   '
      f'({"ALL COVERED" if len(studies)==n_cohort else "!! MISSING "+str(n_cohort-len(studies))})')
assert len(studies) == n_cohort, 'some cohort study has no image embedding'

sp = idx.split.value_counts()
print('by split :', dict(sp))
for s in ['train','val','test']:
    Xs = X[idx.split.values==s]
    print(f'  {s:5s} {len(Xs):6,d}  CLS-norm {np.linalg.norm(Xs,axis=1).mean():.2f}')

# stats 06 will standardize with -- TRAIN ONLY (plan §B1)
mu = X[idx.split.values=='train'].mean(0)
sd = X[idx.split.values=='train'].std(0) + 1e-6
np.savez(os.path.join(CACHE_DIR, 'x_cxr_train_stats.npz'), mean=mu, std=sd)
print('\nsaved train-only standardization stats (mean/std) for 06')
print('\nartifacts in', CACHE_DIR, ':')
for f in ['cxr_cls.npy','cxr_cls_index.parquet','study_x_cxr.npy',
          'study_x_cxr_index.parquet','x_cxr_train_stats.npz']:
    p = os.path.join(CACHE_DIR, f)
    print(f'  {f:28s}', f'{os.path.getsize(p)/1e6:8.1f} MB' if os.path.exists(p) else 'MISSING')
print('\nNEXT: 06_train_eval.ipynb  (EhrEncoder -> FiLM(gamma,beta) -> fuse x_cxr -> DB-Loss head)')

studies in x_cxr : 211,130 / 211,130 cohort   (ALL COVERED)
by split : {'train': np.int64(147791), 'val': np.int64(31670), 'test': np.int64(31669)}
  train 147,791  CLS-norm 13.33
  val   31,670  CLS-norm 13.30
  test  31,669  CLS-norm 13.30

saved train-only standardization stats (mean/std) for 06

artifacts in /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_cls_cache :
  cxr_cls.npy                    1064.7 MB
  cxr_cls_index.parquet            16.3 MB
  study_x_cxr.npy                 648.6 MB
  study_x_cxr_index.parquet         2.3 MB
  x_cxr_train_stats.npz             0.0 MB

NEXT: 06_train_eval.ipynb  (EhrEncoder -> FiLM(gamma,beta) -> fuse x_cxr -> DB-Loss head)
